# TaxGPT — Episode 7: The Feed-Forward (MLP) Block

Companion notebook to blog post *"The Feed-Forward Network (MLP) in Transformers Explained (TaxGPT Episode 7)"*.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 4.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. The feed-forward network: expand, activate, project back down

In [2]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, emb_dim)

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

EMB_DIM, HIDDEN_DIM = 768, 3072   # 4x expansion, GPT-2 convention -- verify against your own spec
ff = FeedForward(EMB_DIM, HIDDEN_DIM)

x = torch.randn(2, 5, EMB_DIM)
out = ff(x)
print("input shape: ", x.shape)
print("output shape:", out.shape, " (same as input -- position-wise, no shape change)")

input shape:  torch.Size([2, 5, 768])
output shape: torch.Size([2, 5, 768])  (same as input -- position-wise, no shape change)


## 2. Confirming it's applied per-token, independently

Change one token's input and verify only that token's output changes — the FFN should not mix information across positions (that's attention's job, not this block's).

In [3]:
x_a = torch.randn(1, 4, EMB_DIM)
x_b = x_a.clone()
x_b[0, 2] = torch.randn(EMB_DIM)  # perturb only position 2

out_a = ff(x_a)
out_b = ff(x_b)

diffs = (out_a - out_b).abs().mean(dim=-1).squeeze(0)
print("mean abs output difference per position:")
for i, d in enumerate(diffs):
    print(f"  position {i}: {d.item():.6f}")
print()
print("Only position 2 should show a meaningful difference -- confirms the FFN doesn't mix across tokens.")

mean abs output difference per position:
  position 0: 0.000000
  position 1: 0.000000
  position 2: 0.211672
  position 3: 0.000000

Only position 2 should show a meaningful difference -- confirms the FFN doesn't mix across tokens.


## 3. GELU vs ReLU: the actual shapes

In [4]:
x_range = torch.linspace(-4, 4, 17)
relu_out = torch.relu(x_range)
gelu_out = F.gelu(x_range)

print(f"{'input':>8} {'ReLU':>10} {'GELU':>10}")
for xi, r, g in zip(x_range, relu_out, gelu_out):
    print(f"{xi.item():8.2f} {r.item():10.4f} {g.item():10.4f}")

   input       ReLU       GELU
   -4.00     0.0000    -0.0001
   -3.50     0.0000    -0.0008
   -3.00     0.0000    -0.0041
   -2.50     0.0000    -0.0155
   -2.00     0.0000    -0.0455
   -1.50     0.0000    -0.1002
   -1.00     0.0000    -0.1587
   -0.50     0.0000    -0.1543
    0.00     0.0000     0.0000
    0.50     0.5000     0.3457
    1.00     1.0000     0.8413
    1.50     1.5000     1.3998
    2.00     2.0000     1.9545
    2.50     2.5000     2.4845
    3.00     3.0000     2.9959
    3.50     3.5000     3.4992
    4.00     4.0000     3.9999


Notice GELU allows small negative inputs (e.g. -1.0, -0.5) to produce small negative outputs, rather than ReLU's hard zero cutoff — that smoothness near zero is the practical reason most modern transformers use GELU over ReLU.

## 4. Where the parameters actually go

In [5]:
ffn_params = sum(p.numel() for p in ff.parameters())

# rough attention params per block for comparison (4 projections, no bias, emb_dim x emb_dim each)
attn_params_approx = 4 * (EMB_DIM * EMB_DIM)

print(f"feed-forward params in this block: {ffn_params:,}")
print(f"attention params per block (approx, Episode 5): {attn_params_approx:,}")
print(f"FFN share of the two combined: {ffn_params / (ffn_params + attn_params_approx):.1%}")

feed-forward params in this block: 4,722,432
attention params per block (approx, Episode 5): 2,359,296
FFN share of the two combined: 66.7%


## Takeaway

The feed-forward network is where genuinely nonlinear, per-token processing happens — attention only ever produces linear combinations of other tokens' Value vectors. The 4x expand-then-contract pattern gives the network room to compute a richer transformation before compressing back to the model's working dimension, and it typically holds the largest parameter share of any component in a single transformer block.

**Next notebook: Episode 8 — The Transformer Block: assembling attention and the feed-forward network.**